# 04 BM25

**Nombre:** Joshua Daniel Menendez Farias

**Fecha:** 11/11/2026

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
import tarfile
import os
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def preprocesar(texto):
    return re.findall(r'\b[a-z]+\b', texto.lower())

def extraer_titulo_gutenberg(texto_raw, doc_id):
    """
    Busca la cabecera 'Title:' típica de Project Gutenberg o toma la primera línea no vacía.
    """
    match = re.search(r'^Title:\s*(.+)$', texto_raw, re.MULTILINE | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    for line in texto_raw.splitlines():
        limpia = line.strip()
        if limpia and not limpia.startswith('***'):
            return limpia[:80]
    return doc_id

def cargar_corpus_grande(ruta_tar, max_docs=300):
    """
    Reutiliza la función de extracción de la práctica 02 (acotada a 300 documentos
    para evitar saturar la memoria RAM), conservando textos originales, títulos y tokens.
    Evita contar distintas versiones del mismo libro filtrando por ID numérico de Gutenberg.
    """
    corpus_tokens = {}
    textos_originales = {}
    titulos = {}
    libros_vistos = set()
    documentos_validos = 0
    
    if not os.path.exists(ruta_tar):
        print(f"Error: El archivo '{ruta_tar}' no se encuentra en el directorio actual.")
        return corpus_tokens, textos_originales, titulos

    with tarfile.open(ruta_tar, 'r') as tar:
        for member in tar:
            if documentos_validos >= max_docs:
                break
            
            if member.isfile() and member.name.endswith('.txt'):
                # Evitar versiones duplicadas del mismo libro (ej. 12345-0.txt, 12345-8.txt)
                nombre_base = os.path.basename(member.name)
                id_libro = nombre_base.split('-')[0].split('.')[0]
                if id_libro in libros_vistos:
                    continue
                
                f = tar.extractfile(member)
                if f:
                    try:
                        texto = f.read().decode('utf-8', errors='ignore')
                        tokens = preprocesar(texto)
                        if tokens:
                            doc_id = member.name
                            corpus_tokens[doc_id] = tokens
                            textos_originales[doc_id] = texto
                            titulos[doc_id] = extraer_titulo_gutenberg(texto, doc_id)
                            libros_vistos.add(id_libro)
                            documentos_validos += 1
                    except Exception:
                        continue
                        
    return corpus_tokens, textos_originales, titulos

# Ejecución sobre los 300 documentos del archivo txt-files.tar
ruta_tar = 'txt-files.tar'
corpus_tokens, textos_originales, titulos = cargar_corpus_grande(ruta_tar, max_docs=300)
docs_ids = list(corpus_tokens.keys())

filas_tabla = []
vocabulario_set = set()
longitudes = []

for doc_id in docs_ids:
    tokens = corpus_tokens[doc_id]
    terminos_unicos = set(tokens)
    vocabulario_set.update(terminos_unicos)
    longitudes.append(len(tokens))
    
    filas_tabla.append({
        'Identificador de documento': doc_id,
        'Título': titulos[doc_id],
        'Texto': textos_originales[doc_id],
        'Tokens': tokens  # Lista de tokens normalizados generada por preprocesar()
    })

df_corpus = pd.DataFrame(filas_tabla)

# Opcional: Evitar que Jupyter/Pandas se trabe o desborde al mostrar textos largos en pantalla
pd.set_option('display.max_colwidth', 100)
display(df_corpus)

if longitudes:
    print(f"\n--- Estadísticas del Corpus ({len(docs_ids)} documentos) ---")
    print(f"Número de documentos: {len(docs_ids)}")
    print(f"Tamaño del vocabulario: {len(vocabulario_set)} términos únicos")
    print(f"Longitud mínima de los documentos: {np.min(longitudes)} tokens")
    print(f"Longitud media de los documentos: {np.mean(longitudes):.2f} tokens")
    print(f"Longitud máxima de los documentos: {np.max(longitudes)} tokens")

,Identificador de documento,Título,Texto,Tokens
0,cache/epub/10000/pg10000.txt,The Magna Carta,The Project Gutenberg eBook of The Magna Carta\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, the, magna, carta, this, ebook, is, for, the, use, of, anyo..."
1,cache/epub/10001/pg10001.txt,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, apocolocyntosis, this, ebook, is, for, the, use, of, anyone..."
2,cache/epub/10002/pg10002.txt,The House on the Borderland,The Project Gutenberg eBook of The House on the Borderland\r\n \r\nThis eBook is for the use ...,"[the, project, gutenberg, ebook, of, the, house, on, the, borderland, this, ebook, is, for, the,..."
3,cache/epub/10003/pg10003.txt,"My First Years as a Frenchwoman, 1876-1879","The Project Gutenberg eBook of My First Years as a Frenchwoman, 1876-1879\r\n \r\nThis eBook ...","[the, project, gutenberg, ebook, of, my, first, years, as, a, frenchwoman, this, ebook, is, for,..."
4,cache/epub/10004/pg10004.txt,The Warriors,The Project Gutenberg eBook of The Warriors\r\n \r\nThis eBook is for the use of anyone anywh...,"[the, project, gutenberg, ebook, of, the, warriors, this, ebook, is, for, the, use, of, anyone, ..."
...,...,...,...,...
295,cache/epub/10388/pg10388.txt,"A Collection of Old English Plays, Volume 1","The Project Gutenberg eBook of A Collection of Old English Plays, Volume 1\r\n \r\nThis eBook...","[the, project, gutenberg, ebook, of, a, collection, of, old, english, plays, volume, this, ebook..."
296,cache/epub/10389/pg10389.txt,"Northern Trails, Book I.","The Project Gutenberg eBook of Northern Trails, Book I.\r\n \r\nThis eBook is for the use of ...","[the, project, gutenberg, ebook, of, northern, trails, book, i, this, ebook, is, for, the, use, ..."
297,cache/epub/1038/pg1038.txt,Style,The Project Gutenberg eBook of Style\r\n \r\nThis eBook is for the use of anyone anywhere in ...,"[the, project, gutenberg, ebook, of, style, this, ebook, is, for, the, use, of, anyone, anywhere..."
298,cache/epub/10390/pg10390.txt,The Edinburgh Lectures on Mental Science,The Project Gutenberg eBook of The Edinburgh Lectures on Mental Science\r\n \r\nThis eBook is...,"[the, project, gutenberg, ebook, of, the, edinburgh, lectures, on, mental, science, this, ebook,..."



--- Estadísticas del Corpus (300 documentos) ---
Número de documentos: 300
Tamaño del vocabulario: 277088 términos únicos
Longitud mínima de los documentos: 5262 tokens
Longitud media de los documentos: 75287.35 tokens
Longitud máxima de los documentos: 985957 tokens


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Asegurar que los doc_ids se traten como cadenas
docs_ids = [str(d) for d in corpus_tokens.keys()]
N = len(docs_ids)

# 1. Cálculo de Document Frequency (df) real sobre los 300 documentos cargados
df_counts = Counter()
for doc_id in docs_ids:
    df_counts.update(set(corpus_tokens[doc_id]))

# 2. Definición de las 8 consultas en inglés (variando longitud y frecuencia de términos)
consultas = [
    {
        "id": "Q1",
        "query": "time",
        "intencion": "Buscar documentos que aborden el concepto general del tiempo (1 término, muy frecuente)."
    },
    {
        "id": "Q2",
        "query": "man world life",
        "intencion": "Recuperar obras sobre la existencia humana y la sociedad en general (3 términos, muy frecuentes)."
    },
    {
        "id": "Q3",
        "query": "whale",
        "intencion": "Localizar textos específicos sobre ballenas o caza marítima (1 término, poco frecuente)."
    },
    {
        "id": "Q4",
        "query": "electricity telegraph",
        "intencion": "Encontrar documentos técnicos o históricos sobre telegrafía y electricidad (2 términos, poco frecuentes)."
    },
    {
        "id": "Q5",
        "query": "king war battle",
        "intencion": "Buscar crónicas históricas o literarias sobre conflictos bélicos y monarquías (3 términos, frecuencia media-alta)."
    },
    {
        "id": "Q6",
        "query": "captain ship sea ocean voyage",
        "intencion": "Recuperar relatos de exploración naval y travesías marítimas (5 términos, combinación frecuente y específica)."
    },
    {
        "id": "Q7",
        "query": "science philosophy nature truth knowledge reason",
        "intencion": "Identificar tratados filosóficos o ensayos científicos sobre el conocimiento humano (6 términos, frecuencia media)."
    },
    {
        "id": "Q8",
        "query": "cryptocurrency blockchain quantum",
        "intencion": "Evaluar el comportamiento ante una consulta sin términos en el vocabulario histórico del corpus (3 términos OOV)."
    }
]

# Imprimir tabla de justificación con df exacto en la colección cargada
print(f"=== JUSTIFICACIÓN DE CONSULTAS MEDIANTE DOCUMENT FREQUENCY (N = {N}) ===")
for c in consultas:
    tokens_q = preprocesar(c["query"])
    dfs = {t: df_counts.get(t, 0) for t in tokens_q}
    print(f"{c['id']}: '{c['query']}' -> {c['intencion']}")
    print(f"    df por término: {dfs}\n")


# ==============================================================================
# MODELO 1: TF-IDF + SIMILITUD COSENO (Conservando representación dispersa)
# ==============================================================================
# Usamos el mismo tokenizador 'preprocesar' para mantener exactamente el mismo vocabulario
vectorizador_tfidf = TfidfVectorizer(tokenizer=preprocesar, token_pattern=None, lowercase=False)
# matriz_tfidf es una scipy.sparse.csr_matrix (dispersa)
matriz_tfidf = vectorizador_tfidf.fit_transform([textos_originales[d] for d in docs_ids])

def buscar_tfidf(consulta_str, top_k=10):
    tokens_q = preprocesar(consulta_str)
    # Si ningún término está en el vocabulario, retornar lista vacía inmediatamente
    if not any(t in vectorizador_tfidf.vocabulary_ for t in tokens_q):
        return []
    
    # Vector disperso de la consulta
    vec_q = vectorizador_tfidf.transform([consulta_str])
    # Producto punto disperso (cosine_similarity devuelve ndarray 1D al usar .ravel())
    scores = cosine_similarity(vec_q, matriz_tfidf).ravel()
    
    # Filtrar score > 0, ordenar por score descendente (-x[1]) y desempatar por doc_id ascendente (x[0])
    resultados = [(docs_ids[i], float(scores[i])) for i in range(N) if scores[i] > 0.0]
    resultados.sort(key=lambda x: (-x[1], x[0]))
    return resultados[:top_k]


# ==============================================================================
# MODELO 2: OKAPI BM25 (k1 = 1.2, b = 0.75, estadísticas precalculadas)
# ==============================================================================
class BM25Index:
    def __init__(self, corpus_tokens, docs_ids, k1=1.2, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs_ids = docs_ids
        self.N = len(docs_ids)
        
        # Estadísticas estáticas de la colección
        self.doc_len = {d: len(corpus_tokens[d]) for d in docs_ids}
        self.avgdl = sum(self.doc_len.values()) / self.N if self.N > 0 else 0.0
        self.tf = {d: Counter(corpus_tokens[d]) for d in docs_ids}
        
        self.df = Counter()
        for d in docs_ids:
            self.df.update(self.tf[d].keys())
            
        # IDF estándar de Robertson-Spärck Jones (variante no negativa)
        self.idf = {
            t: np.log((self.N - freq + 0.5) / (freq + 0.5) + 1.0)
            for t, freq in self.df.items()
        }

    def buscar(self, consulta_str, top_k=10):
        tokens_q = preprocesar(consulta_str)
        # Filtrar solo términos presentes en el vocabulario
        tokens_validos = [t for t in tokens_q if t in self.idf]
        if not tokens_validos:
            return []
        
        resultados = []
        for d in self.docs_ids:
            score = 0.0
            dl = self.doc_len[d]
            tf_doc = self.tf[d]
            
            for t in tokens_validos:
                f = tf_doc.get(t, 0)
                if f > 0:
                    numerador = f * (self.k1 + 1.0)
                    denominador = f + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))
                    score += self.idf[t] * (numerador / denominador)
            
            if score > 0.0:
                resultados.append((d, float(score)))
                
        # Ordenar por puntuación descendente y desempatar por doc_id ascendente (como cadena)
        resultados.sort(key=lambda x: (-x[1], x[0]))
        return resultados[:top_k]

# Instanciar índice BM25 una única vez (estadísticas intactas entre consultas)
bm25 = BM25Index(corpus_tokens, docs_ids, k1=1.2, b=0.75)


# ==============================================================================
# EJECUCIÓN Y PRESENTACIÓN DE RESULTADOS POR CONSULTA
# ==============================================================================
for c in consultas:
    res_tfidf = buscar_tfidf(c["query"], top_k=10)
    res_bm25 = bm25.buscar(c["query"], top_k=10)
    
    print("=" * 80)
    print(f"[{c['id']}] Consulta: '{c['query']}'")
    print(f"Intención: {c['intencion']}")
    
    print("\n  -> Top TF-IDF + Coseno:")
    if not res_tfidf:
        print("     [] (Sin resultados con score positivo)")
    else:
        for rank, (doc_id, sc) in enumerate(res_tfidf, 1):
            print(f"     {rank:2d}. {doc_id} | Score: {sc:.6f} | {titulos[doc_id][:45]}")
            
    print("\n  -> Top BM25 (k1=1.2, b=0.75):")
    if not res_bm25:
        print("     [] (Sin resultados con score positivo)")
    else:
        for rank, (doc_id, sc) in enumerate(res_bm25, 1):
            print(f"     {rank:2d}. {doc_id} | Score: {sc:.6f} | {titulos[doc_id][:45]}")

=== JUSTIFICACIÓN DE CONSULTAS MEDIANTE DOCUMENT FREQUENCY (N = 300) ===
Q1: 'time' -> Buscar documentos que aborden el concepto general del tiempo (1 término, muy frecuente).
    df por término: {'time': 277}

Q2: 'man world life' -> Recuperar obras sobre la existencia humana y la sociedad en general (3 términos, muy frecuentes).
    df por término: {'man': 288, 'world': 300, 'life': 300}

Q3: 'whale' -> Localizar textos específicos sobre ballenas o caza marítima (1 término, poco frecuente).
    df por término: {'whale': 49}

Q4: 'electricity telegraph' -> Encontrar documentos técnicos o históricos sobre telegrafía y electricidad (2 términos, poco frecuentes).
    df por término: {'electricity': 48, 'telegraph': 72}

Q5: 'king war battle' -> Buscar crónicas históricas o literarias sobre conflictos bélicos y monarquías (3 términos, frecuencia media-alta).
    df por término: {'king': 230, 'war': 235, 'battle': 199}

Q6: 'captain ship sea ocean voyage' -> Recuperar relatos de exploració

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
from IPython.display import display, Markdown

# ==============================================================================
# PARTE 3A: TABLAS COMPARATIVAS Y MÉTRICAS DE SOLAPAMIENTO POR CONSULTA
# ==============================================================================
resultados_globales = {}

for c in consultas:
    q_id = c["id"]
    query_str = c["query"]
    
    res_tfidf = buscar_tfidf(query_str, top_k=10)
    res_bm25 = bm25.buscar(query_str, top_k=10)
    
    resultados_globales[q_id] = {
        "query": query_str,
        "tfidf": res_tfidf,
        "bm25": res_bm25
    }
    
    # Conjuntos de IDs para calcular la intersección
    ids_tfidf = [doc_id for doc_id, _ in res_tfidf]
    ids_bm25 = [doc_id for doc_id, _ in res_bm25]
    interseccion = len(set(ids_tfidf) & set(ids_bm25))
    
    top1_tfidf = f"`{ids_tfidf[0]}` ({titulos[ids_tfidf[0]][:40]})" if ids_tfidf else "Ninguno (lista vacía)"
    top1_bm25 = f"`{ids_bm25[0]}` ({titulos[ids_bm25[0]][:40]})" if ids_bm25 else "Ninguno (lista vacía)"
    
    # Construcción de la tabla Markdown
    md_lines = [
        f"### {q_id}: `{query_str}`",
        f"*{c['intencion']}*\n",
        f"- **Tamaño real de las listas:** $|T_{{\\mathrm{{TFIDF}}}}| = {len(res_tfidf)}$, $|T_{{\\mathrm{{BM25}}}}| = {len(res_bm25)}$",
        f"- **IDs compartidos en Top 10:** $|T_{{\\mathrm{{TFIDF}}}} \\cap T_{{\\mathrm{{BM25}}}}| = {interseccion}$",
        f"- **Primer documento (Posición 1):** TF-IDF $\\rightarrow$ {top1_tfidf} | BM25 $\\rightarrow$ {top1_bm25}\n",
        "| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |",
        "|---:|---|---:|---|---:|"
    ]
    
    max_filas = max(len(res_tfidf), len(res_bm25))
    if max_filas == 0:
        md_lines.append("| - | *Sin resultados (vocabulario OOV)* | 0.000000 | *Sin resultados (vocabulario OOV)* | 0.000000 |")
    else:
        for i in range(max_filas):
            if i < len(res_tfidf):
                d_tf, sc_tf = res_tfidf[i]
                # Limpiar caracteres '|' en títulos para no romper la tabla Markdown
                tit_tf = titulos[d_tf][:38].replace("|", "-")
                col_tf = f"`{d_tf}` - {tit_tf}"
                val_tf = f"{sc_tf:.6f}"
            else:
                col_tf, val_tf = "—", "—"
                
            if i < len(res_bm25):
                d_bm, sc_bm = res_bm25[i]
                tit_bm = titulos[d_bm][:38].replace("|", "-")
                col_bm = f"`{d_bm}` - {tit_bm}"
                val_bm = f"{sc_bm:.6f}"
            else:
                col_bm, val_bm = "—", "—"
                
            md_lines.append(f"| {i+1} | {col_tf} | {val_tf} | {col_bm} | {val_bm} |")
            
    display(Markdown("\n".join(md_lines)))


# ==============================================================================
# PARTE 3B: ANÁLISIS MICROSCÓPICO DE CAMBIO DE POSICIÓN (2 DOCUMENTOS)
# ==============================================================================
def analizar_divergencia(q_id, doc_a=None, doc_b=None):
    """
    Extrae |d|, f(t,d), df(t) y el aporte individual de cada término en BM25 y TF-IDF
    para explicar por qué dos documentos cambian de orden o distancia entre rankings.
    Si doc_a y doc_b son None, busca automáticamente dos documentos que hayan cambiado
    de posición relativa dentro de la consulta indicada.
    """
    data = resultados_globales[q_id]
    query_str = data["query"]
    res_tfidf = data["tfidf"]
    res_bm25 = data["bm25"]
    
    rank_tf = {d: (idx + 1, sc) for idx, (d, sc) in enumerate(res_tfidf)}
    rank_bm = {d: (idx + 1, sc) for idx, (d, sc) in enumerate(res_bm25)}
    
    # Selección automática de un par con inversión de orden o cambio fuerte de distancia
    if doc_a is None or doc_b is None:
        comunes = list(set(rank_tf.keys()) & set(rank_bm.keys()))
        mejor_par = None
        max_cambio = -1
        for i in range(len(comunes)):
            for j in range(i + 1, len(comunes)):
                d1, d2 = comunes[i], comunes[j]
                diff_tf = rank_tf[d1][0] - rank_tf[d2][0]
                diff_bm = rank_bm[d1][0] - rank_bm[d2][0]
                # Priorizar inversiones de orden (signos opuestos) o gran cambio de distancia
                cambio = abs(diff_tf - diff_bm) + (10 if diff_tf * diff_bm < 0 else 0)
                if cambio > max_cambio:
                    max_cambio = cambio
                    mejor_par = (d1, d2)
        if mejor_par:
            doc_a, doc_b = mejor_par
        else:
            print("No hay suficientes documentos comunes en esta consulta para comparar posiciones.")
            return

    tokens_q = preprocesar(query_str)
    # Eliminar duplicados manteniendo orden de aparición
    terminos_unicos_q = list(dict.fromkeys(tokens_q))
    
    # Obtener pesos exactos de la matriz dispersa TF-IDF para los documentos seleccionados
    vocab = vectorizador_tfidf.vocabulary_
    idx_doc_a = docs_ids.index(doc_a)
    idx_doc_b = docs_ids.index(doc_b)
    
    md_analisis = [
        f"## Análisis de divergencia de ranking: Consulta {q_id} (`{query_str}`)",
        f"- **Longitud media del corpus ($\\text{{avgdl}}$):** `{bm25.avgdl:.2f}` tokens ($N = {bm25.N}$)\n",
        f"### Documentos comparados:",
        f"1. **Doc A (`{doc_a}`):** *{titulos[doc_a]}*",
        f"   - Longitud $|d_A| = {bm25.doc_len[doc_a]}$ tokens (ratio $|d_A|/\\text{{avgdl}} = {bm25.doc_len[doc_a]/bm25.avgdl:.2f}$)",
        f"   - Posición TF-IDF: **#{rank_tf.get(doc_a, ('>10', 0))[0]}** (Coseno = `{rank_tf.get(doc_a, (0, 0.0))[1]:.6f}`)",
        f"   - Posición BM25: **#{rank_bm.get(doc_a, ('>10', 0))[0]}** (Score = `{rank_bm.get(doc_a, (0, 0.0))[1]:.6f}`)\n",
        f"2. **Doc B (`{doc_b}`):** *{titulos[doc_b]}*",
        f"   - Longitud $|d_B| = {bm25.doc_len[doc_b]}$ tokens (ratio $|d_B|/\\text{{avgdl}} = {bm25.doc_len[doc_b]/bm25.avgdl:.2f}$)",
        f"   - Posición TF-IDF: **#{rank_tf.get(doc_b, ('>10', 0))[0]}** (Coseno = `{rank_tf.get(doc_b, (0, 0.0))[1]:.6f}`)",
        f"   - Posición BM25: **#{rank_bm.get(doc_b, ('>10', 0))[0]}** (Score = `{rank_bm.get(doc_b, (0, 0.0))[1]:.6f}`)\n",
        "### Desglose término a término",
        "| Término $t$ | $df(t)$ | $\\text{IDF}_{\\text{BM25}}(t)$ | $f(t, d_A)$ | Aporte BM25 ($d_A$) | Peso TF-IDF ($d_A$) | $f(t, d_B)$ | Aporte BM25 ($d_B$) | Peso TF-IDF ($d_B$) |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|"
    ]
    
    for t in terminos_unicos_q:
        df_t = bm25.df.get(t, 0)
        idf_t = bm25.idf.get(t, 0.0)
        
        # Cálculos Doc A
        fa = bm25.tf[doc_a].get(t, 0)
        dl_a = bm25.doc_len[doc_a]
        bm_a = idf_t * ((fa * (bm25.k1 + 1)) / (fa + bm25.k1 * (1 - bm25.b + bm25.b * (dl_a / bm25.avgdl)))) if fa > 0 else 0.0
        tfidf_a = matriz_tfidf[idx_doc_a, vocab[t]] if t in vocab else 0.0
        
        # Cálculos Doc B
        fb = bm25.tf[doc_b].get(t, 0)
        dl_b = bm25.doc_len[doc_b]
        bm_b = idf_t * ((fb * (bm25.k1 + 1)) / (fb + bm25.k1 * (1 - bm25.b + bm25.b * (dl_b / bm25.avgdl)))) if fb > 0 else 0.0
        tfidf_b = matriz_tfidf[idx_doc_b, vocab[t]] if t in vocab else 0.0
        
        md_analisis.append(
            f"| `{t}` | {df_t} | {idf_t:.4f} | {fa} | {bm_a:.4f} | {tfidf_a:.4f} | {fb} | {bm_b:.4f} | {tfidf_b:.4f} |"
        )
        
    display(Markdown("\n".join(md_analisis)))

# Ejecutar análisis automático sobre la consulta multitérmino Q6 (puedes cambiarla por Q5, Q7, etc.)
analizar_divergencia("Q6")

### Q1: `time`
*Buscar documentos que aborden el concepto general del tiempo (1 término, muy frecuente).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 7$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10075/pg10075.txt` (A Narrative of the Life and Adventures o) | BM25 $\rightarrow$ `cache/epub/10378/pg10378.txt` (Autobiography)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10075/pg10075.txt` - A Narrative of the Life and Adventures | 0.039378 | `cache/epub/10378/pg10378.txt` - Autobiography | 0.178012 |
| 2 | `cache/epub/10267/pg10267.txt` - The Outdoor Chums; Or, The First Tour  | 0.034089 | `cache/epub/10146/pg10146.txt` - Reminiscences of Pioneer Days in St. P | 0.177912 |
| 3 | `cache/epub/10378/pg10378.txt` - Autobiography | 0.033028 | `cache/epub/10030/pg10030.txt` - The Life of Hon. William F. Cody, Know | 0.177892 |
| 4 | `cache/epub/10368/pg10368.txt` - The Vizier of the Two-Horned Alexander | 0.032711 | `cache/epub/10369/pg10369.txt` - Memoir and Diary of John Yeardley, Min | 0.177888 |
| 5 | `cache/epub/10211/pg10211.txt` - At Whispering Pine Lodge | 0.031614 | `cache/epub/10136/pg10136.txt` - The Book of Household Management | 0.177862 |
| 6 | `cache/epub/10146/pg10146.txt` - Reminiscences of Pioneer Days in St. P | 0.029965 | `cache/epub/10162/pg10162.txt` - Dio's Rome, Volume 3 | 0.177860 |
| 7 | `cache/epub/10086/pg10086.txt` - The Minute Boys of the Mohawk Valley | 0.028857 | `cache/epub/10129/pg10129.txt` - Excellent Women | 0.177845 |
| 8 | `cache/epub/10030/pg10030.txt` - The Life of Hon. William F. Cody, Know | 0.028684 | `cache/epub/10086/pg10086.txt` - The Minute Boys of the Mohawk Valley | 0.177844 |
| 9 | `cache/epub/10002/pg10002.txt` - The House on the Borderland | 0.027948 | `cache/epub/10267/pg10267.txt` - The Outdoor Chums; Or, The First Tour  | 0.177787 |
| 10 | `cache/epub/10162/pg10162.txt` - Dio's Rome, Volume 3 | 0.027773 | `cache/epub/10002/pg10002.txt` - The House on the Borderland | 0.177785 |

### Q2: `man world life`
*Recuperar obras sobre la existencia humana y la sociedad en general (3 términos, muy frecuentes).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 5$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10004/pg10004.txt` (The Warriors) | BM25 $\rightarrow$ `cache/epub/10004/pg10004.txt` (The Warriors)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10004/pg10004.txt` - The Warriors | 0.074754 | `cache/epub/10004/pg10004.txt` - The Warriors | 0.100306 |
| 2 | `cache/epub/1017/pg1017.txt` - The Soul of Man under Socialism | 0.058003 | `cache/epub/10116/pg10116.txt` - All Saints' Day and Other Sermons | 0.100301 |
| 3 | `cache/epub/10116/pg10116.txt` - All Saints' Day and Other Sermons | 0.057210 | `cache/epub/10325/pg10325.txt` - The Gospel of the Pentateuch: A Set of | 0.100268 |
| 4 | `cache/epub/10326/pg10326.txt` - David: Five Sermons | 0.053900 | `cache/epub/10391/pg10391.txt` - The Wolf's Long Howl | 0.100128 |
| 5 | `cache/epub/10325/pg10325.txt` - The Gospel of the Pentateuch: A Set of | 0.050439 | `cache/epub/10132/pg10132.txt` - The Sowers | 0.100112 |
| 6 | `cache/epub/10063/pg10063.txt` - Happiness and Marriage | 0.049083 | `cache/epub/1017/pg1017.txt` - The Soul of Man under Socialism | 0.100110 |
| 7 | `cache/epub/10328/pg10328.txt` - Poems | 0.040176 | `cache/epub/10324/pg10324.txt` - Bull Hunter | 0.100092 |
| 8 | `cache/epub/10291/pg10291.txt` - In the Fourth Year: Anticipations of a | 0.036313 | `cache/epub/10357/pg10357.txt` - Life of Johnson, Volume 4 | 0.100090 |
| 9 | `cache/epub/10324/pg10324.txt` - Bull Hunter | 0.034390 | `cache/epub/10364/pg10364.txt` - Yeast: a Problem | 0.100088 |
| 10 | `cache/epub/10274/pg10274.txt` - Sermons on Evil-Speaking | 0.034283 | `cache/epub/10213/pg10213.txt` - The Everlasting Whisper | 0.100071 |

### Q3: `whale`
*Localizar textos específicos sobre ballenas o caza marítima (1 término, poco frecuente).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 7$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10015/pg10015.txt` (Punchinello, Volume 1, No. 19, August 6,) | BM25 $\rightarrow$ `cache/epub/10015/pg10015.txt` (Punchinello, Volume 1, No. 19, August 6,)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10015/pg10015.txt` - Punchinello, Volume 1, No. 19, August  | 0.021714 | `cache/epub/10015/pg10015.txt` - Punchinello, Volume 1, No. 19, August  | 3.812631 |
| 2 | `cache/epub/10040/pg10040.txt` - Aleutian Indian and English Dictionary | 0.015892 | `cache/epub/10224/pg10224.txt` - Kalitan, Our Little Alaskan Cousin | 3.802055 |
| 3 | `cache/epub/10224/pg10224.txt` - Kalitan, Our Little Alaskan Cousin | 0.013151 | `cache/epub/10122/pg10122.txt` - Fairies and Fusiliers | 3.582074 |
| 4 | `cache/epub/10122/pg10122.txt` - Fairies and Fusiliers | 0.011082 | `cache/epub/10040/pg10040.txt` - Aleutian Indian and English Dictionary | 3.542745 |
| 5 | `cache/epub/1018/pg1018.txt` - The Water-Babies | 0.003389 | `cache/epub/1018/pg1018.txt` - The Water-Babies | 3.516857 |
| 6 | `cache/epub/10367/pg10367.txt` - Poems | 0.002451 | `cache/epub/10339/pg10339.txt` - An Antarctic Mystery | 3.351032 |
| 7 | `cache/epub/10075/pg10075.txt` - A Narrative of the Life and Adventures | 0.002169 | `cache/epub/10367/pg10367.txt` - Poems | 3.053483 |
| 8 | `cache/epub/10339/pg10339.txt` - An Antarctic Mystery | 0.002046 | `cache/epub/10121/pg10121.txt` - The Literature of Arabia | 3.029751 |
| 9 | `cache/epub/10105/pg10105.txt` - Punchinello, Volume 2, No. 33, Novembe | 0.001775 | `cache/epub/102/pg102.txt` - The Tragedy of Pudd'nhead Wilson | 2.977194 |
| 10 | `cache/epub/10036/pg10036.txt` - Punchinello, Volume 2, No. 28, October | 0.001761 | `cache/epub/10130/pg10130.txt` - The Works of Charles and Mary Lamb — V | 2.839216 |

### Q4: `electricity telegraph`
*Encontrar documentos técnicos o históricos sobre telegrafía y electricidad (2 términos, poco frecuentes).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 7$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10059/pg10059.txt` (Aunt Jane's Nieces on Vacation) | BM25 $\rightarrow$ `cache/epub/10043/pg10043.txt` (Klondyke Nuggets: A Brief Description of)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10059/pg10059.txt` - Aunt Jane's Nieces on Vacation | 0.006438 | `cache/epub/10043/pg10043.txt` - Klondyke Nuggets: A Brief Description  | 6.347168 |
| 2 | `cache/epub/10043/pg10043.txt` - Klondyke Nuggets: A Brief Description  | 0.005361 | `cache/epub/10059/pg10059.txt` - Aunt Jane's Nieces on Vacation | 6.213311 |
| 3 | `cache/epub/10146/pg10146.txt` - Reminiscences of Pioneer Days in St. P | 0.005285 | `cache/epub/10165/pg10165.txt` - Across the Zodiac | 6.052425 |
| 4 | `cache/epub/10128/pg10128.txt` - The Great Events by Famous Historians, | 0.004121 | `cache/epub/10128/pg10128.txt` - The Great Events by Famous Historians, | 5.933953 |
| 5 | `cache/epub/10390/pg10390.txt` - The Edinburgh Lectures on Mental Scien | 0.003609 | `cache/epub/10251/pg10251.txt` - Town Geology | 5.855173 |
| 6 | `cache/epub/101/pg101.txt` - The Hacker Crackdown: Law and Disorder | 0.003193 | `cache/epub/10146/pg10146.txt` - Reminiscences of Pioneer Days in St. P | 5.804161 |
| 7 | `cache/epub/10251/pg10251.txt` - Town Geology | 0.003121 | `cache/epub/101/pg101.txt` - The Hacker Crackdown: Law and Disorder | 5.116008 |
| 8 | `cache/epub/10165/pg10165.txt` - Across the Zodiac | 0.002833 | `cache/epub/10065/pg10065.txt` - The Constitution of the United States | 4.730189 |
| 9 | `cache/epub/10035/pg10035.txt` - Punchinello, Volume 2, No. 27, October | 0.002817 | `cache/epub/10391/pg10391.txt` - The Wolf's Long Howl | 4.545780 |
| 10 | `cache/epub/10106/pg10106.txt` - Punchinello, Volume 2, No. 34, Novembe | 0.001931 | `cache/epub/1024/pg1024.txt` - The Wrecker | 4.543307 |

### Q5: `king war battle`
*Buscar crónicas históricas o literarias sobre conflictos bélicos y monarquías (3 términos, frecuencia media-alta).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 1$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/1030/pg1030.txt` (The Cavalier Songs and Ballads of Englan) | BM25 $\rightarrow$ `cache/epub/10114/pg10114.txt` (The Great Events by Famous Historians, V)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/1030/pg1030.txt` - The Cavalier Songs and Ballads of Engl | 0.047254 | `cache/epub/10114/pg10114.txt` - The Great Events by Famous Historians, | 2.008935 |
| 2 | `cache/epub/10213/pg10213.txt` - The Everlasting Whisper | 0.045653 | `cache/epub/10315/pg10315.txt` - The Persian Literature, Comprising The | 2.004176 |
| 3 | `cache/epub/10090/pg10090.txt` - Peaceless Europe | 0.041357 | `cache/epub/10151/pg10151.txt` - The Great Events by Famous Historians, | 2.002742 |
| 4 | `cache/epub/10159/pg10159.txt` - England and the War | 0.040400 | `cache/epub/10147/pg10147.txt` - America's War for Humanity | 1.999639 |
| 5 | `cache/epub/10333/pg10333.txt` - The European Anarchy | 0.040027 | `cache/epub/10162/pg10162.txt` - Dio's Rome, Volume 3 | 1.992631 |
| 6 | `cache/epub/10147/pg10147.txt` - America's War for Humanity | 0.038802 | `cache/epub/10103/pg10103.txt` - The Great Events by Famous Historians, | 1.987552 |
| 7 | `cache/epub/10166/pg10166.txt` - What Germany Thinks; Or, The War as Ge | 0.034820 | `cache/epub/10128/pg10128.txt` - The Great Events by Famous Historians, | 1.972751 |
| 8 | `cache/epub/10097/pg10097.txt` - The Healing of Nations and the Hidden  | 0.032597 | `cache/epub/10024/pg10024.txt` - Beneath the Banner: Being Narratives o | 1.962996 |
| 9 | `cache/epub/100/pg100.txt` - The Complete Works of William Shakespe | 0.032424 | `cache/epub/10341/pg10341.txt` - The Great Events by Famous Historians, | 1.962525 |
| 10 | `cache/epub/10099/pg10099.txt` - Towards the Goal | 0.030548 | `cache/epub/10085/pg10085.txt` - Moorish Literature | 1.961146 |

### Q6: `captain ship sea ocean voyage`
*Recuperar relatos de exploración naval y travesías marítimas (5 términos, combinación frecuente y específica).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 7$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10339/pg10339.txt` (An Antarctic Mystery) | BM25 $\rightarrow$ `cache/epub/10339/pg10339.txt` (An Antarctic Mystery)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10339/pg10339.txt` - An Antarctic Mystery | 0.056520 | `cache/epub/10339/pg10339.txt` - An Antarctic Mystery | 5.417652 |
| 2 | `cache/epub/10008/pg10008.txt` - The Mystery | 0.044299 | `cache/epub/10103/pg10103.txt` - The Great Events by Famous Historians, | 5.314872 |
| 3 | `cache/epub/10370/pg10370.txt` - Sustained honor: The Age of Liberty Es | 0.041223 | `cache/epub/10387/pg10387.txt` - A Century Too Soon: The Age of Tyranny | 5.305073 |
| 4 | `cache/epub/10081/pg10081.txt` - The Boy Allies at Jutland; Or, The Gre | 0.028418 | `cache/epub/10210/pg10210.txt` - Wolves of the Sea | 5.304612 |
| 5 | `cache/epub/1024/pg1024.txt` - The Wrecker | 0.026583 | `cache/epub/10008/pg10008.txt` - The Mystery | 5.258502 |
| 6 | `cache/epub/10210/pg10210.txt` - Wolves of the Sea | 0.021695 | `cache/epub/10005/pg10005.txt` - A Voyage to the Moon | 5.228175 |
| 7 | `cache/epub/10387/pg10387.txt` - A Century Too Soon: The Age of Tyranny | 0.017387 | `cache/epub/10370/pg10370.txt` - Sustained honor: The Age of Liberty Es | 5.222389 |
| 8 | `cache/epub/10121/pg10121.txt` - The Literature of Arabia | 0.015770 | `cache/epub/10121/pg10121.txt` - The Literature of Arabia | 5.214552 |
| 9 | `cache/epub/10109/pg10109.txt` - The Unspeakable Gentleman | 0.015647 | `cache/epub/1024/pg1024.txt` - The Wrecker | 5.204412 |
| 10 | `cache/epub/10359/pg10359.txt` - Aunt Jane's Nieces at Millville | 0.015593 | `cache/epub/10350/pg10350.txt` - The Works of Samuel Johnson, LL.D. in  | 5.181479 |

### Q7: `science philosophy nature truth knowledge reason`
*Identificar tratados filosóficos o ensayos científicos sobre el conocimiento humano (6 términos, frecuencia media).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 4$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10214/pg10214.txt` (Introduction to the Philosophy and Writi) | BM25 $\rightarrow$ `cache/epub/10214/pg10214.txt` (Introduction to the Philosophy and Writi)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10214/pg10214.txt` - Introduction to the Philosophy and Wri | 0.033312 | `cache/epub/10214/pg10214.txt` - Introduction to the Philosophy and Wri | 4.237024 |
| 2 | `cache/epub/10139/pg10139.txt` - The Faith of the Millions (2nd series) | 0.026394 | `cache/epub/10139/pg10139.txt` - The Faith of the Millions (2nd series) | 4.234976 |
| 3 | `cache/epub/10390/pg10390.txt` - The Edinburgh Lectures on Mental Scien | 0.026308 | `cache/epub/10378/pg10378.txt` - Autobiography | 4.171737 |
| 4 | `cache/epub/10314/pg10314.txt` - The Sonnets of Michael Angelo Buonarro | 0.019548 | `cache/epub/10361/pg10361.txt` - The Creative Process in the Individual | 4.134001 |
| 5 | `cache/epub/10361/pg10361.txt` - The Creative Process in the Individual | 0.018895 | `cache/epub/10088/pg10088.txt` - Thaumaturgia; Or, Elucidations of the  | 4.115670 |
| 6 | `cache/epub/10274/pg10274.txt` - Sermons on Evil-Speaking | 0.017513 | `cache/epub/10009/pg10009.txt` - Wild Northern Scenes; Or, Sporting Adv | 4.083963 |
| 7 | `cache/epub/1022/pg1022.txt` - Walking | 0.016793 | `cache/epub/10314/pg10314.txt` - The Sonnets of Michael Angelo Buonarro | 4.078588 |
| 8 | `cache/epub/10375/pg10375.txt` - England's Antiphon | 0.014717 | `cache/epub/1038/pg1038.txt` - Style | 4.061235 |
| 9 | `cache/epub/10251/pg10251.txt` - Town Geology | 0.014509 | `cache/epub/10350/pg10350.txt` - The Works of Samuel Johnson, LL.D. in  | 4.052105 |
| 10 | `cache/epub/10078/pg10078.txt` - Plain Words from America: A Letter to  | 0.014077 | `cache/epub/10004/pg10004.txt` - The Warriors | 4.030599 |

### Q8: `cryptocurrency blockchain quantum`
*Evaluar el comportamiento ante una consulta sin términos en el vocabulario histórico del corpus (3 términos OOV).*

- **Tamaño real de las listas:** $|T_{\mathrm{TFIDF}}| = 10$, $|T_{\mathrm{BM25}}| = 10$
- **IDs compartidos en Top 10:** $|T_{\mathrm{TFIDF}} \cap T_{\mathrm{BM25}}| = 10$
- **Primer documento (Posición 1):** TF-IDF $\rightarrow$ `cache/epub/10054/pg10054.txt` (Nova analysis aquarum Medeviensium) | BM25 $\rightarrow$ `cache/epub/10054/pg10054.txt` (Nova analysis aquarum Medeviensium)

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | `cache/epub/10054/pg10054.txt` - Nova analysis aquarum Medeviensium | 0.018927 | `cache/epub/10054/pg10054.txt` - Nova analysis aquarum Medeviensium | 6.574947 |
| 2 | `cache/epub/10386/pg10386.txt` - Thoughts on the Necessity of Improving | 0.001143 | `cache/epub/10058/pg10058.txt` - The Divine Office: A Study of the Roma | 4.633963 |
| 3 | `cache/epub/10058/pg10058.txt` - The Divine Office: A Study of the Roma | 0.000900 | `cache/epub/10386/pg10386.txt` - Thoughts on the Necessity of Improving | 4.347272 |
| 4 | `cache/epub/10266/pg10266.txt` - The Glands Regulating Personality | 0.000529 | `cache/epub/10266/pg10266.txt` - The Glands Regulating Personality | 3.960364 |
| 5 | `cache/epub/10161/pg10161.txt` - English Poets of the Eighteenth Centur | 0.000463 | `cache/epub/10343/pg10343.txt` - The Works of Charles and Mary Lamb — V | 3.202859 |
| 6 | `cache/epub/10343/pg10343.txt` - The Works of Charles and Mary Lamb — V | 0.000454 | `cache/epub/10088/pg10088.txt` - Thaumaturgia; Or, Elucidations of the  | 3.034076 |
| 7 | `cache/epub/10088/pg10088.txt` - Thaumaturgia; Or, Elucidations of the  | 0.000390 | `cache/epub/10161/pg10161.txt` - English Poets of the Eighteenth Centur | 2.883179 |
| 8 | `cache/epub/10357/pg10357.txt` - Life of Johnson, Volume 4 | 0.000225 | `cache/epub/10223/pg10223.txt` - Meyers Konversationslexikon Band 15 | 2.685897 |
| 9 | `cache/epub/10055/pg10055.txt` - Hamburgische Dramaturgie | 0.000130 | `cache/epub/10055/pg10055.txt` - Hamburgische Dramaturgie | 2.500797 |
| 10 | `cache/epub/10223/pg10223.txt` - Meyers Konversationslexikon Band 15 | 0.000128 | `cache/epub/10357/pg10357.txt` - Life of Johnson, Volume 4 | 2.005255 |

## Análisis de divergencia de ranking: Consulta Q6 (`captain ship sea ocean voyage`)
- **Longitud media del corpus ($\text{avgdl}$):** `75287.35` tokens ($N = 300$)

### Documentos comparados:
1. **Doc A (`cache/epub/1024/pg1024.txt`):** *The Wrecker*
   - Longitud $|d_A| = 145321$ tokens (ratio $|d_A|/\text{avgdl} = 1.93$)
   - Posición TF-IDF: **#5** (Coseno = `0.026583`)
   - Posición BM25: **#9** (Score = `5.204412`)

2. **Doc B (`cache/epub/10387/pg10387.txt`):** *A Century Too Soon: The Age of Tyranny*
   - Longitud $|d_B| = 86829$ tokens (ratio $|d_B|/\text{avgdl} = 1.15$)
   - Posición TF-IDF: **#7** (Coseno = `0.017387`)
   - Posición BM25: **#3** (Score = `5.305073`)

### Desglose término a término
| Término $t$ | $df(t)$ | $\text{IDF}_{\text{BM25}}(t)$ | $f(t, d_A)$ | Aporte BM25 ($d_A$) | Peso TF-IDF ($d_A$) | $f(t, d_B)$ | Aporte BM25 ($d_B$) | Peso TF-IDF ($d_B$) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| `captain` | 165 | 0.5981 | 307 | 1.3072 | 0.0323 | 58 | 1.2862 | 0.0101 |
| `ship` | 196 | 0.4264 | 162 | 0.9265 | 0.0152 | 101 | 0.9259 | 0.0158 |
| `sea` | 251 | 0.1797 | 121 | 0.3887 | 0.0094 | 60 | 0.3866 | 0.0078 |
| `ocean` | 185 | 0.4841 | 17 | 0.9510 | 0.0017 | 19 | 0.9949 | 0.0031 |
| `voyage` | 130 | 0.8357 | 16 | 1.6310 | 0.0019 | 18 | 1.7114 | 0.0036 |

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [5]:
# 3. Tabla comparativa usando DataFrame de Pandas (renderizado limpio y alineado)
filas_pos = []
for d in docs_a_rastrear:
    dl = modelos_bm25["Base"].doc_len[d]
    ratio = dl / modelos_bm25["Base"].avgdl
    tit = titulos[d][:40]
    
    b_info = rankings_completos["Base"].get(d, {"pos": "-", "score": 0.0})
    nl_info = rankings_completos["Sin ajuste por longitud"].get(d, {"pos": "-", "score": 0.0})
    mr_info = rankings_completos["Mayor efecto de repeticiones"].get(d, {"pos": "-", "score": 0.0})
    
    filas_pos.append({
        "ID Documento": d,
        "Título": tit,
        "|d|": dl,
        "Ratio |d|/avgdl": f"{ratio:.2f}x",
        "Base (k1=1.2, b=0.75)": f"#{b_info['pos']} ({b_info['score']:.4f})",
        "Sin Longitud (k1=1.2, b=0.00)": f"#{nl_info['pos']} ({nl_info['score']:.4f})",
        "Mayor Rep. (k1=2.0, b=0.75)": f"#{mr_info['pos']} ({mr_info['score']:.4f})"
    })

df_posiciones = pd.DataFrame(filas_pos)
print(f"Comparación de posiciones y scores — Consulta {q_id_param} ('{query_str}')")
print(f"Longitud media del corpus (avgdl): {modelos_bm25['Base'].avgdl:.2f} tokens\n")
display(df_posiciones)

# 4. Desglose de aportes término a término para los primeros 4 documentos
def aporte_termino(modelo, doc_id, termino):
    f = modelo.tf[doc_id].get(termino, 0)
    if f == 0 or termino not in modelo.idf:
        return 0.0
    dl = modelo.doc_len[doc_id]
    num = f * (modelo.k1 + 1.0)
    den = f + modelo.k1 * (1.0 - modelo.b + modelo.b * (dl / modelo.avgdl))
    return modelo.idf[termino] * (num / den)

filas_aportes = []
for d in docs_a_rastrear[:4]:
    dl = modelos_bm25["Base"].doc_len[d]
    for t in tokens_q:
        f_td = modelos_bm25["Base"].tf[d].get(t, 0)
        filas_aportes.append({
            "ID Documento": d,
            "|d|": dl,
            "Término": t,
            "f(t,d)": f_td,
            "Aporte Base (1.2, 0.75)": round(aporte_termino(modelos_bm25["Base"], d, t), 4),
            "Aporte Sin Long. (1.2, 0.00)": round(aporte_termino(modelos_bm25["Sin ajuste por longitud"], d, t), 4),
            "Aporte Mayor Rep. (2.0, 0.75)": round(aporte_termino(modelos_bm25["Mayor efecto de repeticiones"], d, t), 4)
        })

df_aportes = pd.DataFrame(filas_aportes)
print(f"\nDesglose de aportes por término (primeros 4 documentos)")
display(df_aportes)

Comparación de posiciones y scores — Consulta Q6 ('captain ship sea ocean voyage')
Longitud media del corpus (avgdl): 75287.35 tokens



,ID Documento,Título,|d|,Ratio |d|/avgdl,"Base (k1=1.2, b=0.75)","Sin Longitud (k1=1.2, b=0.00)","Mayor Rep. (k1=2.0, b=0.75)"
0,cache/epub/10339/pg10339.txt,An Antarctic Mystery,83292,1.11x,#1 (5.4177),#1 (5.4272),#1 (7.2737)
1,cache/epub/10103/pg10103.txt,"The Great Events by Famous Historians, V",164033,2.18x,#2 (5.3149),#2 (5.4232),#2 (7.0503)
2,cache/epub/10387/pg10387.txt,A Century Too Soon: The Age of Tyranny,86829,1.15x,#3 (5.3051),#7 (5.3293),#4 (7.0298)
3,cache/epub/10210/pg10210.txt,Wolves of the Sea,98326,1.31x,#4 (5.3046),#5 (5.3480),#3 (7.0346)
4,cache/epub/10008/pg10008.txt,The Mystery,70837,0.94x,#5 (5.2585),#10 (5.2461),#5 (6.9402)
5,cache/epub/10350/pg10350.txt,"The Works of Samuel Johnson, LL.D. in El",180960,2.40x,#10 (5.1815),#4 (5.3637),#10 (6.7726)
6,cache/epub/100/pg100.txt,The Complete Works of William Shakespear,985957,13.10x,#36 (4.4040),#3 (5.3992),#37 (5.3615)



Desglose de aportes por término (primeros 4 documentos)


,ID Documento,|d|,Término,"f(t,d)","Aporte Base (1.2, 0.75)","Aporte Sin Long. (1.2, 0.00)","Aporte Mayor Rep. (2.0, 0.75)"
0,cache/epub/10339/pg10339.txt,83292,captain,465,1.3122,1.3125,1.7861
1,cache/epub/10339/pg10339.txt,83292,ship,118,0.9280,0.9287,1.2564
2,cache/epub/10339/pg10339.txt,83292,sea,147,0.3918,0.3921,0.5312
3,cache/epub/10339/pg10339.txt,83292,ocean,18,0.9934,0.9984,1.2966
4,cache/epub/10339/pg10339.txt,83292,voyage,50,1.7922,1.7955,2.4034
5,cache/epub/10103/pg10103.txt,164033,captain,48,1.2567,1.2838,1.6638
6,cache/epub/10103/pg10103.txt,164033,ship,53,0.8998,0.9174,1.1944
7,cache/epub/10103/pg10103.txt,164033,sea,120,0.3880,0.3914,0.5226
8,cache/epub/10103/pg10103.txt,164033,ocean,24,0.9732,1.0142,1.2551
9,cache/epub/10103/pg10103.txt,164033,voyage,98,1.7972,1.8164,2.4144
